In [22]:
# Import required libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Lambda, Dropout
import matplotlib.pyplot as plt

# Our extended training text
text = """The cat sat on the mat and watched the mouse.
          The dog ran in the park and played with the ball.
          A cat and dog play in the garden together.
          The bird flew over the tree and landed on a branch.
          The mouse ran under the table and hid from the cat.
          A dog chased the ball in the garden happily.
          The cat jumped on the table and drank some milk.
          Birds sing in the trees and build their nests.
          Children play in the park with their dogs.
          The mouse found cheese on the kitchen floor."""

# Preprocess text: make lowercase and remove punctuation
text = text.lower()
text = text.replace('.', '')
words = text.split()

print("Our words:", words)
print("Total words:", len(words))
print("Unique words:", len(set(words)))

Our words: ['the', 'cat', 'sat', 'on', 'the', 'mat', 'and', 'watched', 'the', 'mouse', 'the', 'dog', 'ran', 'in', 'the', 'park', 'and', 'played', 'with', 'the', 'ball', 'a', 'cat', 'and', 'dog', 'play', 'in', 'the', 'garden', 'together', 'the', 'bird', 'flew', 'over', 'the', 'tree', 'and', 'landed', 'on', 'a', 'branch', 'the', 'mouse', 'ran', 'under', 'the', 'table', 'and', 'hid', 'from', 'the', 'cat', 'a', 'dog', 'chased', 'the', 'ball', 'in', 'the', 'garden', 'happily', 'the', 'cat', 'jumped', 'on', 'the', 'table', 'and', 'drank', 'some', 'milk', 'birds', 'sing', 'in', 'the', 'trees', 'and', 'build', 'their', 'nests', 'children', 'play', 'in', 'the', 'park', 'with', 'their', 'dogs', 'the', 'mouse', 'found', 'cheese', 'on', 'the', 'kitchen', 'floor']
Total words: 96
Unique words: 47


In [27]:
# Create word-to-index mapping
tokenizer = Tokenizer()
tokenizer.fit_on_texts([' '.join(words)])
vocab_size = len(tokenizer.word_index) + 1

# We'll use 2 words before and after (window_size = 2)
window_size = 2

def create_training_data(words, window_size):
    X, y = [], []  # X: context words, y: target word
    
    for i in range(window_size, len(words) - window_size):
        # Get context words (2 before + 2 after)
        context = (words[i-2:i] + words[i+1:i+3])
        target = words[i]
        
        # Convert words to indices
        context_indices = [tokenizer.word_index[word] for word in context]
        target_index = tokenizer.word_index[target]
        
        X.append(context_indices)
        y.append(target_index)
    
    return np.array(X), np.array(y)

# Generate training data
X, y = create_training_data(words, window_size)

In [24]:
# Build an improved model
embedding_size = 100  # Increased embedding size
context_size = window_size * 2  # Total context words (2 before + 2 after = 4)

model = Sequential([
    # Embedding layer with more dimensions
    Embedding(vocab_size, embedding_size, input_length=context_size),
    
    # First Dense layer after embedding
    Lambda(lambda x: tf.reduce_mean(x, axis=1)),
    Dense(128, activation='relu'),
    Dropout(0.2),  # Add dropout to prevent overfitting
    
    # Second Dense layer
    Dense(64, activation='relu'),
    Dropout(0.2),
    
    # Output layer
    Dense(vocab_size, activation='softmax')
])

# Compile with a lower learning rate for better convergence
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer,
             loss='sparse_categorical_crossentropy',
             metrics=['accuracy'])

# Show model structure
print("Model Summary:")
model.summary()

Model Summary:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [28]:
# Train the model with more epochs and validation
history = model.fit(X, y,
                   epochs=100,  # Increased epochs
                   batch_size=32,
                   validation_split=0.2,  # Added validation
                   verbose=1)
                   

# Function to predict word given context words
def predict_word(context_words):
    # Convert words to indices
    if len(context_words) != 4:
        print("Please provide exactly 4 context words!")
        return
    
    try:
        # Convert words to their indices
        context_indices = [tokenizer.word_index[word.lower()] for word in context_words]
        # Convert to numpy array with correct shape
        context_array = np.array([context_indices])
        # Make prediction
        prediction = model.predict(context_array, verbose=0)[0]  # Reduced verbosity
        # Get top 5 predictions with probabilities
        top_indices = prediction.argsort()[-5:][::-1]
        
        print(f"\nContext: {' '.join(context_words[:2])} ___ {' '.join(context_words[2:])}")
        print("\nTop 5 predicted words:")
        for i in top_indices:
            word = list(tokenizer.word_index.keys())[list(tokenizer.word_index.values()).index(i)]
            prob = prediction[i]
            print(f"{word}: {prob:.4f}")
            
    except KeyError as e:
        print(f"Word not in vocabulary! Please use words from the training text.")

# Test with examples from our training data
print("Example 1 (from training):")
predict_word(['the', 'cat', 'on', 'the'])

print("\nExample 2 (from training):")
predict_word(['the', 'dog', 'in', 'the'])

print("\nExample 3 (from training):")
predict_word(['mouse', 'ran', 'the', 'table'])

Epoch 1/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.7527 - loss: 0.8481 - val_accuracy: 0.0526 - val_loss: 7.0764
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8690 - loss: 0.5763 - val_accuracy: 0.0526 - val_loss: 7.1225
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8065 - loss: 0.6397 - val_accuracy: 0.0526 - val_loss: 7.1614
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8202 - loss: 0.7404 - val_accuracy: 0.0526 - val_loss: 7.2024
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8661 - loss: 0.5979 - val_accuracy: 0.0526 - val_loss: 7.2538
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8612 - loss: 0.5363 - val_accuracy: 0.0526 - val_loss: 7.3088
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.8035 - loss: 0.6848 - val_accuracy: 0.0526 - val_loss: 7.3378
Epoch 8/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.8436 - loss: 0.6168 - val_accuracy: 0.0526 - val_loss: